In [1]:
from __future__ import annotations

import os
# os.environ["HWLOC_HIDE_ERRORS"] = "1"
# os.environ["TF_ENABLE_NUMA"] = "0"
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"
# 3. Provide valid temporary directory for ptxas CUDA kernel compilation
os.environ["TMPDIR"] = "/tmp"
os.environ["TMP"] = "/tmp"
os.environ["TEMP"] = "/tmp"

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
print("Jax version", jax.__version__, "64-bit", jax.config.read("jax_enable_x64"))
print("jax", jax.__version__, "GPU backend:", jax.default_backend() == 'gpu', f"({jax.devices('gpu')[0].device_kind})" if jax.default_backend() == 'gpu' else "")
import jax.numpy as jnp
import s2fft
from tqdm.auto import tqdm
from typing import Literal

import jax.numpy as jnp
import s2fft
import argparse
import shutil
from pathlib import Path

def resample_mw_to_mwss(map_path: Path, output_path: Path) -> None:
    m = np.load(map_path)
    m = m.reshape((1500, 2999, 5)).astype(np.float32)

    m_mwss = np.stack(
        [
            np.array(s2fft.utils.resampling_jax.mw_to_mwss(m[..., c], L=1500)).real
            for c in range(5)
        ],
        axis=-1,
    )

    np.save(output_path, m_mwss.astype(np.float64, copy=False))

def L_downsample_mwss(map_path: Path, output_path: Path, L_in: int, L_out: int) -> None:
    m = np.load(map_path)
    # m_halfL = np.stack(
    #         [
    #             np.array(s2fft.utils.resampling.downsample_by_two_mwss(m[..., c], L=L_in)).real
    #             for c in range(5)
    #         ], 
    #     axis=-1
    #     )  # shape: (L_in/2, ...)
    # m_halfL = np.array(s2fft.utils.resampling.downsample_by_two_mwss(m, L=L_in)).real
    # m = m[0:-1:2, 0:-1:2, :]
    def L_downsample(f_map, L_in, L_out):
        L_cut = L_in-L_out
        m = s2fft.transforms.spherical.forward(f_map, L=L_in, reality=True, method='jax_cuda', sampling='mwss')
        m = m[:L_out, L_cut:-L_cut]  # keep modes l < 750
        m = s2fft.transforms.spherical.inverse(m, L=L_out, reality=True, method='jax_cuda', sampling='mwss')
        return m
    m = np.stack(
            [
                np.array(L_downsample(m[..., c], L_in=L_in, L_out=L_out)).real
                for c in range(5)
            ], 
        axis=-1
        )  # shape: (L_in/2, ...)

    np.save(output_path, m.astype(np.float64, copy=False))
    del m
    # raise NotImplementedError("L_downsample_by_two_mwss is not implemented yet.")



Jax version 0.11.0 64-bit True


E0922 22:08:33.759255  204309 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]


jax 0.11.0 GPU backend: True (NVIDIA GH200 120GB)


Downsample

In [3]:
SOURCE_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_1500_batch_3_mwss/"
)
OUTPUT_DIR = Path(
    "/projects/u6pf/brianycc/spherical_maps/"
    "5_tomo_bins_flm_750_batch_3_mwss/"
)
OVERWRITE = False


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    params_src = SOURCE_DIR / "params.npy"
    params_dst = OUTPUT_DIR / "params.npy"
    if params_src.exists() and (OVERWRITE or not params_dst.exists()):
        shutil.copy2(params_src, params_dst)

    map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
    if not map_files:
        raise FileNotFoundError(f"No map_*.npy files found in {SOURCE_DIR}")
    
    converted = 0
    skipped = 0

    for map_path in tqdm(map_files, desc="Resampling maps"):
        output_path = OUTPUT_DIR / map_path.name
        if output_path.exists() and not OVERWRITE:
            skipped += 1
            continue

        # resample_mw_to_mwss(map_path, output_path)
        L_downsample_mwss(map_path, output_path, L_in=1500, L_out=750)
        converted += 1

    print(
        f"Converted {converted} maps, skipped {skipped} existing maps.\n"
        f"Source: {SOURCE_DIR}\n"
        f"Output: {OUTPUT_DIR}"
    )

print("SOURCE_DIR =", SOURCE_DIR)
print("Exists     =", SOURCE_DIR.exists())
print("Is dir     =", SOURCE_DIR.is_dir())
map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
print("Count      =", len(map_files))
print("First 5    =", [p.name for p in map_files[:5]])

main()

SOURCE_DIR = /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_3_mwss
Exists     = True
Is dir     = True
Count      = 2000
First 5    = ['map_0.npy', 'map_1.npy', 'map_10.npy', 'map_100.npy', 'map_1000.npy']


Resampling maps:   0%|                                                                                                  | 0/2000 [00:00<?, ?it/s]

Resampling maps: 100%|█████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [2:23:19<00:00,  4.30s/it]

Converted 2000 maps, skipped 0 existing maps.
Source: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_1500_batch_3_mwss
Output: /projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_batch_3_mwss


Compute data stats


In [2]:
import os
import glob
import numpy as np
import jax.numpy as jnp
from tqdm import tqdm

def compute_and_save_map_stats(directory, output_filename="map_stats.npz"):
    """
    Computes channel-wise mean and std across all map_{i}.npy files in `directory`
    loading only one map into memory at a time. Saves stats into `output_filename`.
    """
    map_files = glob.glob(os.path.join(directory, "map_*.npy"))
    n_files = len(map_files)
    assert n_files > 0, f"No map_*.npy files found in {directory}"

    print(f"Found {n_files} map files. Calculating mean and std...")

    # Inspect shape of first map to initialize accumulators
    sample_map = jnp.load(map_files[0])
    num_channels = sample_map.shape[-1]
    
    # Online accumulators for each channel
    count = 0
    mean = jnp.zeros(num_channels, dtype=jnp.float32)
    M2 = jnp.zeros(num_channels, dtype=jnp.float32)

    # Welford's algorithm over each map loading only 1 map at a time
    for map_path in tqdm(map_files):
        m = jnp.load(map_path).astype(jnp.float32)
        # Collapse spatial dimensions (theta, phi) -> reshape to (-1, channels)
        pixels = m.reshape(-1, num_channels)
        
        n_pixels = pixels.shape[0]
        batch_mean = jnp.mean(pixels, axis=0)
        batch_var = jnp.var(pixels, axis=0)

        # Update global accumulators (Welford / Parallel algorithm)
        if count == 0:
            mean = batch_mean
            M2 = batch_var * n_pixels
            count = n_pixels
        else:
            delta = batch_mean - mean
            new_count = count + n_pixels
            mean += delta * n_pixels / new_count
            M2 += batch_var * n_pixels + (delta ** 2) * count * n_pixels / new_count
            count = new_count

    variance = M2 / count
    std = jnp.sqrt(variance)

    # Cast stats back to float32 for model training
    mean = mean.astype(jnp.float32)
    std = std.astype(jnp.float32)

    stats_path = os.path.join(directory, output_filename)
    jnp.savez(stats_path, mean=mean, std=std)
    print(f"Successfully saved stats to {stats_path}")
    print("Channel Mean:", mean)
    print("Channel Std :", std)
    return stats_path

# Execute calculation for your map directory
# data_dir = "/projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_all_mwss/"
# data_dir = "/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_augmented_pooled"
# data_dir = "/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mw_samples_holdout"
data_dir = "/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_holdout"
compute_and_save_map_stats(data_dir)


Found 16384 map files. Calculating mean and std...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16384/16384 [20:58<00:00, 13.02it/s]

Successfully saved stats to /projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_holdout/map_stats.npz
Channel Mean: [-0.00015975 -0.00015953 -0.00015991 -0.00015978 -0.00015994]
Channel Std : [0.0461393  0.04614866 0.04615127 0.04614862 0.0461506 ]


'/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_holdout/map_stats.npz'

MW to MWSS resampling

In [2]:
SOURCE_DIR = Path(
    # "/projects/u6pf/brianycc/spherical_maps/"
    # "map_compression_samples_5_tomo_bins_flm_1500_batch_3/"
    # "/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mw_samples_augmented_pooled"
    "/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mw_samples_holdout"
)
OUTPUT_DIR = Path(
    # "/projects/u6pf/brianycc/spherical_maps/"
    # "5_tomo_bins_flm_1500_batch_3_mwss/"
    # "/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_augmented_pooled"
    "/projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_holdout"
)
SKIP = []


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Source directory : {SOURCE_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")

    params_src = SOURCE_DIR / "params.npy"
    params_dst = OUTPUT_DIR / "params.npy"
    if params_src.exists():
        if params_dst.exists():
            print(f"params.npy already exists at {params_dst}; skipping copy.")
        else:
            print(f"Copying params.npy -> {params_dst}")
            shutil.copy2(params_src, params_dst)
    else:
        print(f"Warning: {params_src} not found.")

    map_files = sorted(SOURCE_DIR.glob("map_*.npy"))
    if not map_files:
        raise FileNotFoundError(f"No map_*.npy files found in {SOURCE_DIR}")

    print(f"Found {len(map_files)} map files to process.")

    skipped = 0
    converted = 0

    for i, map_path in enumerate(
        tqdm(map_files, desc="Resampling maps", unit="map", leave=True),
        start=1,
    ):
        output_path = OUTPUT_DIR / map_path.name
        if output_path.exists():
            skipped += 1
            print(f"[{i}/{len(map_files)}] Skipping existing file: {map_path.name}")
            continue
        elif map_path.name in SKIP:
            skipped += 1
            print(f"[{i}/{len(map_files)}] Skipping file in SKIP list: {map_path.name}")
            continue

        # print(f"[{i}/{len(map_files)}] Converting {map_path.name} -> {output_path.name}")
        resample_mw_to_mwss(map_path, output_path)
        converted += 1
        # print(f"  Done. Converted so far: {converted}, skipped so far: {skipped}")

    print(
        f"Finished. Converted {converted} maps, skipped {skipped} existing maps.\n"
        f"Source: {SOURCE_DIR}\n"
        f"Output: {OUTPUT_DIR}"
    )

main()

Source directory : /projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mw_samples_holdout
Output directory: /projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_holdout
params.npy already exists at /projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_holdout/params.npy; skipping copy.
Found 16384 map files to process.


Resampling maps:   0%|                                                                                                                                                | 0/16384 [00:00<?, ?map/s]

[1/16384] Skipping existing file: map_0.npy
[2/16384] Skipping existing file: map_1.npy
[3/16384] Skipping existing file: map_10.npy
[4/16384] Skipping existing file: map_100.npy
[5/16384] Skipping existing file: map_1000.npy
[6/16384] Skipping existing file: map_10000.npy
[7/16384] Skipping existing file: map_10001.npy
[8/16384] Skipping existing file: map_10002.npy
[9/16384] Skipping existing file: map_10003.npy
[10/16384] Skipping existing file: map_10004.npy
[11/16384] Skipping existing file: map_10005.npy
[12/16384] Skipping existing file: map_10006.npy
[13/16384] Skipping existing file: map_10007.npy
[14/16384] Skipping existing file: map_10008.npy


Resampling maps: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16384/16384 [1:46:48<00:00,  2.56map/s]

Finished. Converted 16370 maps, skipped 14 existing maps.
Source: /projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mw_samples_holdout
Output: /projects/u6pf/brianycc/spherical_maps/map_compression_L1500_mwss_samples_holdout


In [ ]:
# def main() -> None:
#     parser = argparse.ArgumentParser(
#         description="Bulk-resample spherical maps from MW to MWSS."
#     )
#     parser.add_argument(
#         "--source-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1"
#         ),
#         help="Directory containing map_*.npy and params.npy",
#     )
#     parser.add_argument(
#         "--output-dir",
#         type=Path,
#         default=Path(
#             "/projects/u6pf/brianycc/spherical_maps/"
#             "map_compression_samples_5_tomo_bins_flm_1500_batch_1_mwss"
#         ),
#         help="Directory where MWSS-resampled files will be written",
#     )
#     args = parser.parse_args()

#     source_dir: Path = args.source_dir
#     output_dir: Path = args.output_dir
#     output_dir.mkdir(parents=True, exist_ok=True)

#     params_src = source_dir / "params.npy"
#     params_dst = output_dir / "params.npy"
#     if params_src.exists() and not params_dst.exists():
#         shutil.copy2(params_src, params_dst)

#     map_files = sorted(source_dir.glob("map_*.npy"))
#     if not map_files:
#         raise FileNotFoundError(f"No map_*.npy files found in {source_dir}")

#     skipped = 0
#     converted = 0

#     for map_path in map_files:
#         output_path = output_dir / map_path.name
#         if output_path.exists():
#             skipped += 1
#             continue
#         resample_mw_to_mwss(map_path, output_path)
#         converted += 1

#     print(
#         f"Converted {converted} maps, skipped {skipped} existing maps "
#         f"from {source_dir} to {output_dir}"
#     )


# if __name__ == "__main__":
#     main()